# Keras Tuner
Tuning a small network on the Pima diabetes data: the optimizer, the number of nodes, the number of layers,
and finally everything at once. Tuning uses a validation set; the test set is used once, at the very end.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
os.environ["CUDA_VISIBLE_DEVICES"] = ""       # CPU only, so the numbers repeat
import tempfile
import numpy as np
import pandas as pd
import keras
import keras_tuner as kt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.config.experimental.enable_op_determinism()   # same numbers on every run
print("keras", keras.__version__, " keras_tuner", kt.__version__)

## The data: 768 patients, 8 features, diabetes yes or no

In [ ]:
df = pd.read_csv("data/diabetes.csv")
print(df.shape)
print(df.head())
print(df.corr()["Outcome"].round(2))

## Three sets: train 60%, validation 20%, test 20%
The tuner compares models on the validation set. The test set stays untouched until the last cell.

In [ ]:
X = df.drop(columns="Outcome").to_numpy()
y = df["Outcome"].to_numpy()
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, stratify=y_rest, random_state=1)
scaler = StandardScaler().fit(X_train)            # learn the scaling from the training set only
X_train, X_val, X_test = (scaler.transform(a).astype("float32") for a in (X_train, X_val, X_test))
print("train", X_train.shape, " validation", X_val.shape, " test", X_test.shape)
print("share with diabetes in validation:", y_val.mean().round(3))

## The hand-made baseline: one hidden layer of 32 ReLU nodes, Adam, 100 epochs

In [ ]:
def baseline(seed):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(8,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

base = baseline(0)
h = base.fit(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_val, y_val), verbose=0).history
print(f"baseline: validation accuracy after 100 epochs {h['val_accuracy'][-1]:.3f}")

## Step 1: choose the optimizer
`hp.Choice` lists the values to try. Four optimizers, so at most four different models exist; asking for 5 trials
makes the random search stop after 4, when it can find no new combination.

In [ ]:
SEARCH_EPOCHS = 10
work = tempfile.mkdtemp()                         # the tuner writes its files here

def build_model(hp):
    model = keras.Sequential([
        keras.Input(shape=(8,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")])
    optimizer = hp.Choice("optimizer", values=["adam", "sgd", "rmsprop", "adadelta"])
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

keras.utils.set_random_seed(0)
tuner = kt.RandomSearch(build_model, objective="val_accuracy", max_trials=5, seed=0,
                        directory=work, project_name="optimizer", overwrite=True)
tuner.search(X_train, y_train, epochs=SEARCH_EPOCHS, validation_data=(X_val, y_val), verbose=0)
for t in tuner.oracle.get_best_trials(num_trials=10):
    print(t.trial_id, t.hyperparameters.values, round(t.score, 3))
best_optimizer = tuner.get_best_hyperparameters()[0].values["optimizer"]
print("best:", tuner.get_best_hyperparameters()[0].values)

The best model comes back already trained (weights of its best epoch). We continue its training to epoch 100.
Epochs are counted from 0, so the next one is number `SEARCH_EPOCHS`.

In [ ]:
model = tuner.get_best_models(num_models=1)[0]
h = model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=SEARCH_EPOCHS,
              validation_data=(X_val, y_val), verbose=0).history
print(f"epochs run: {len(h['loss'])}  validation accuracy at epoch 100: {h['val_accuracy'][-1]:.3f}")

## Step 2: the number of nodes in the hidden layer, `hp.Int` from 8 to 128 in steps of 8

In [ ]:
def build_model(hp):
    units = hp.Int("units", min_value=8, max_value=128, step=8)
    model = keras.Sequential([
        keras.Input(shape=(8,)),
        keras.layers.Dense(units, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer=best_optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

keras.utils.set_random_seed(0)
tuner = kt.RandomSearch(build_model, objective="val_accuracy", max_trials=5, seed=0,
                        directory=work, project_name="units", overwrite=True)
tuner.search(X_train, y_train, epochs=SEARCH_EPOCHS, validation_data=(X_val, y_val), verbose=0)
for t in tuner.oracle.get_best_trials(num_trials=10):
    print(t.trial_id, t.hyperparameters.values, round(t.score, 3))
best_units = tuner.get_best_hyperparameters()[0].values["units"]
print("best:", tuner.get_best_hyperparameters()[0].values)

Without `step`, every whole number between the limits can be drawn:

In [ ]:
hp = kt.HyperParameters()
hp.Int("units", min_value=8, max_value=128)
print(hp.space[0].get_config())

## Step 3: the number of hidden layers, a loop over `hp.Int("num_layers", 1, 10)`

In [ ]:
def build_model(hp):
    model = keras.Sequential([keras.Input(shape=(8,))])
    for i in range(hp.Int("num_layers", min_value=1, max_value=10)):
        model.add(keras.layers.Dense(best_units, activation="relu"))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=best_optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

keras.utils.set_random_seed(0)
tuner = kt.RandomSearch(build_model, objective="val_accuracy", max_trials=5, seed=0,
                        directory=work, project_name="num_layers", overwrite=True)
tuner.search(X_train, y_train, epochs=SEARCH_EPOCHS, validation_data=(X_val, y_val), verbose=0)
for t in tuner.oracle.get_best_trials(num_trials=10):
    print(t.trial_id, t.hyperparameters.values, round(t.score, 3))
print("best:", tuner.get_best_hyperparameters()[0].values)

## Step 4: everything at once
Number of layers; for every layer its own nodes, activation and dropout rate (names end in the layer number);
and the optimizer. Every trial now trains for the same 100 epochs as the baseline, so the scores compare fairly.

In [ ]:
def build_model(hp):
    model = keras.Sequential([keras.Input(shape=(8,))])
    for i in range(hp.Int("num_layers", min_value=1, max_value=4)):
        model.add(keras.layers.Dense(
            hp.Int(f"units_{i}", min_value=8, max_value=128, step=8),
            activation=hp.Choice(f"activation_{i}", values=["relu", "tanh"])))
        model.add(keras.layers.Dropout(
            hp.Choice(f"dropout_{i}", values=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5])))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    optimizer = hp.Choice("optimizer", values=["adam", "rmsprop", "nadam", "sgd"])
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

keras.utils.set_random_seed(0)
tuner = kt.RandomSearch(build_model, objective="val_accuracy", max_trials=20, seed=0,
                        directory=work, project_name="final", overwrite=True)
tuner.search(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_val, y_val), verbose=0)
rows = []
for t in tuner.oracle.get_best_trials(num_trials=20):
    v = t.hyperparameters.values
    n = v["num_layers"]
    layers = ", ".join(f"{v[f'units_{i}']} {v[f'activation_{i}']} d{v[f'dropout_{i}']}" for i in range(n))
    rows.append(dict(trial=t.trial_id, val_accuracy=round(t.score, 4), num_layers=n,
                     optimizer=v["optimizer"], layers=layers))
trials = pd.DataFrame(rows)
trials.to_csv("data/trials.csv", index=False)
pd.set_option("display.width", 200, "display.max_colwidth", 90)
print(trials)
best_hp = tuner.get_best_hyperparameters()[0]
print("best:", best_hp.values)

Every trial is saved as a folder with a `trial.json` file inside `directory/project_name`:

In [ ]:
folder = os.path.join(work, "final")
print(sorted(os.listdir(folder))[:5], "...")
print(open(os.path.join(folder, "trial_00", "trial.json")).read()[:400])

## Is the tuned network really better? Retrain both from scratch, 5 seeds each
A trial's score is its best epoch out of 100, from one random start, on 154 validation patients. We retrain the
baseline and the tuned configuration 5 times each for 100 epochs and compare the means on the validation set.
Only then, once, the test set.

In [ ]:
def tuned(seed):
    keras.utils.set_random_seed(seed)
    return build_model(best_hp)

results = []
for name, make in (("baseline", baseline), ("tuned", tuned)):
    for seed in range(5):
        m = make(seed)
        m.fit(X_train, y_train, batch_size=32, epochs=100, verbose=0)
        val = m.evaluate(X_val, y_val, verbose=0)[1]
        test = m.evaluate(X_test, y_test, verbose=0)[1]
        results.append(dict(model=name, seed=seed, val_accuracy=val, test_accuracy=test))
res = pd.DataFrame(results)
res.to_csv("data/retrain.csv", index=False)
print(res.round(3))
print(res.groupby("model")[["val_accuracy", "test_accuracy"]].agg(["mean", "std"]).round(3))

# A bigger test: MNIST
On Pima the gain from tuning was within seed noise. On handwritten digits, where the size of the network
and the learning rate matter more, we repeat the experiment: 10,000 training images, 10,000 validation images,
and the 10,000 test images used once, at the end.

In [ ]:
(x, y), (x_te, y_te) = keras.datasets.mnist.load_data()
x = x.reshape(-1, 784).astype("float32") / 255             # pixels from 0-255 to 0-1
x_te = x_te.reshape(-1, 784).astype("float32") / 255
mx_train, my_train = x[:10000], y[:10000]                  # 10,000 training images
mx_val, my_val = x[50000:], y[50000:]                      # 10,000 other images for validation
mx_test, my_test = x_te, y_te                              # the official test images, used once
print(mx_train.shape, mx_val.shape, mx_test.shape)
EPOCHS = 10

The hand-made guess: one hidden layer of 32 ReLU nodes, Adam with its default learning rate 0.001.

In [ ]:
def mnist_baseline(seed):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(784,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

The search space: 1 to 3 layers, 32 to 512 nodes per layer, a dropout rate per layer, the optimizer,
and the learning rate drawn on a log scale between 0.0001 and 0.01.

In [ ]:
def build_mnist(hp):
    model = keras.Sequential([keras.Input(shape=(784,))])
    for i in range(hp.Int("num_layers", 1, 3)):
        model.add(keras.layers.Dense(hp.Int(f"units_{i}", 32, 512, step=32), activation="relu"))
        model.add(keras.layers.Dropout(hp.Choice(f"dropout_{i}", values=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5])))
    model.add(keras.layers.Dense(10, activation="softmax"))
    lr = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")
    name = hp.Choice("optimizer", values=["adam", "rmsprop", "sgd"])
    optimizer = {"adam": keras.optimizers.Adam, "rmsprop": keras.optimizers.RMSprop,
                 "sgd": keras.optimizers.SGD}[name](learning_rate=lr)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

keras.utils.set_random_seed(0)
tuner = kt.RandomSearch(build_mnist, objective="val_accuracy", max_trials=20, executions_per_trial=1,
                        seed=0, directory=work, project_name="mnist", overwrite=True)
tuner.search(mx_train, my_train, batch_size=128, epochs=EPOCHS, validation_data=(mx_val, my_val), verbose=0)
rows = []
for t in tuner.oracle.get_best_trials(num_trials=20):
    v = t.hyperparameters.values
    n = v["num_layers"]
    rows.append(dict(trial=t.trial_id, val_accuracy=round(t.score, 4), num_layers=n, optimizer=v["optimizer"],
                     learning_rate=v["learning_rate"],
                     layers=", ".join(f"{v[f'units_{i}']} d{v[f'dropout_{i}']}" for i in range(n))))
mtrials = pd.DataFrame(rows)
mtrials.to_csv("data/mnist_trials.csv", index=False)
print(mtrials)
mnist_best = tuner.get_best_hyperparameters()[0]
print("best:", {k: v for k, v in mnist_best.values.items()})

Retrain the baseline and the winner from scratch, 5 seeds each, same 10 epochs; then the test set, once.

In [ ]:
results = []
for name, make in (("baseline", mnist_baseline), ("tuned", lambda s: (keras.utils.set_random_seed(s), build_mnist(mnist_best))[1])):
    for seed in range(5):
        m = make(seed)
        m.fit(mx_train, my_train, batch_size=128, epochs=EPOCHS, verbose=0)
        results.append(dict(model=name, seed=seed, val_accuracy=m.evaluate(mx_val, my_val, verbose=0)[1],
                            test_accuracy=m.evaluate(mx_test, my_test, verbose=0)[1]))
mres = pd.DataFrame(results)
mres.to_csv("data/mnist_retrain.csv", index=False)
print(mres.round(4))
print(mres.groupby("model")[["val_accuracy", "test_accuracy"]].agg(["mean", "std"]).round(4))